# 2 — Predicting Directly: Framework, Target, and Why Raw TDA Snapshots Don't Work

Notebook 1 found that regime-label association was the wrong question.
This notebook sets up the approach that replaced it: **predict an
actionable market outcome directly**, and test whether adding TDA
features measurably improves that prediction versus financial features
alone -- a much more direct, harder-to-fool test than regime agreement.

It also walks through the first, humbling result: the original 140 raw
TDA descriptors, added directly, do not help.


In [1]:
import sys
from pathlib import Path

import pandas as pd

DATA = Path.cwd().parent / "data"


## The target: `vol_regime_h7`

Every confirmed finding in this project, from here to the end, is on one
target: **will Ethereum's realized volatility over the next 7 days
exceed its own trailing 90-day level?** A binary classification of a
*volatility regime* -- not price direction, not return magnitude. That
distinction matters a lot and is stated here up front: every attempt in
this project to predict price direction either failed outright or was
actively hurt by the same features that help volatility prediction.

## The framework

- **Arm A** (baseline): 26 standard financial indicators -- returns,
  volatility, momentum, volume, trend.
- **Arm B**: Arm A + the original 140 raw TDA descriptors (per-layer
  entropy, landscape/silhouette norms, connectivity/fragility indices,
  day-over-day distance -- one number per descriptor per day, i.e. a
  single-day *snapshot*, not a trend).
- **Validation**: purged, embargoed walk-forward cross-validation --
  every fold's training window ends well before its test window begins,
  with enough margin that no forward-looking label construction can leak
  across the boundary.
- **Significance**: paired bootstrap test on the Arm-B-minus-Arm-A metric
  gap, with Benjamini-Hochberg false-discovery-rate correction applied
  across every (target, model) comparison tested together.


In [2]:
mc = pd.read_parquet(DATA / "processed/backtest_results/model_comparison_results.parquet")
raw_tda = mc[mc["comparison"] == "arm_b_vs_arm_a"]

n_sig = raw_tda["significant_fdr"].sum()
print(f"{n_sig} / {len(raw_tda)} comparisons significant after FDR correction")
print()
print(raw_tda[raw_tda["significant_fdr"]][["target", "model", "metric", "observed_diff", "p_value_fdr"]].to_string(index=False))


6 / 24 comparisons significant after FDR correction

       target         model   metric  observed_diff  p_value_fdr
fwd_return_h3       xgboost neg_rmse       0.009410     0.000000
fwd_return_h3 random_forest neg_rmse       0.003201     0.000000
fwd_return_h3         ridge neg_rmse      -0.015735     0.000000
   up_h3_tau2 random_forest  roc_auc      -0.037828     0.026182
   up_h7_tau2 random_forest  roc_auc      -0.047988     0.000000
   up_h3_tau1 random_forest  roc_auc      -0.047992     0.009600


Six comparisons reach significance -- and the classification ones are
all **negative**: adding the raw 140 TDA columns makes RandomForest
significantly *worse* at predicting `up_h3_tau1`, `up_h3_tau2`, and
`up_h7_tau2` (directional targets). One positive regression result
survives (`fwd_return_h3`), modest and, as later placebo testing showed,
mostly explained by added model capacity rather than genuine signal.

**On `vol_regime_h7` itself -- the target that matters for everything
that follows -- raw TDA shows no significant effect either direction:**


In [3]:
print(raw_tda[raw_tda["target"] == "vol_regime_h7"][["model", "metric", "observed_diff", "p_value_fdr"]].to_string(index=False))


        model  metric  observed_diff  p_value_fdr
random_forest roc_auc       0.022962        0.156
        dummy roc_auc       0.000000        1.000
       logreg roc_auc      -0.002726        1.000
      xgboost roc_auc      -0.006834        1.000


## Five remediation attempts, all null

Rather than accept "raw TDA doesn't work" after one try, five different
ways of taming the 140 columns were tested: PCA compression to 8
components, regularizing the tree models (shallower depth, higher
leaf-size minimums), univariate feature selection (keep only the
columns most correlated with the target), and L1/Lasso joint selection
at two different sparsity levels. All five came back null or
partial -- summarized in the table below (see `STATUS.md` for the full
detail on each).


In [4]:
lasso = pd.read_parquet(DATA / "processed/backtest_results/model_comparison_results_lasso.parquet")
lasso_sig = lasso[lasso["significant_fdr"]]
print(f"L1/Lasso joint selection (sparsest, genuinely-sparse setting): "
      f"{len(lasso_sig)} / {len(lasso)} significant, none from lasso_logreg/elasticnet themselves")
print(lasso_sig[["target", "model", "metric", "observed_diff", "p_value_fdr"]].to_string(index=False))


L1/Lasso joint selection (sparsest, genuinely-sparse setting): 6 / 30 significant, none from lasso_logreg/elasticnet themselves
       target         model   metric  observed_diff  p_value_fdr
fwd_return_h3       xgboost neg_rmse       0.009410        0.000
fwd_return_h3 random_forest neg_rmse       0.003201        0.000
fwd_return_h3         ridge neg_rmse      -0.015735        0.000
   up_h3_tau2 random_forest  roc_auc      -0.037828        0.030
   up_h7_tau2 random_forest  roc_auc      -0.047988        0.000
   up_h3_tau1 random_forest  roc_auc      -0.047992        0.012


## Conclusion

Six different ways of using the original TDA descriptor set -- raw, and
five different remediation strategies -- were tried, through the same
rigorous walk-forward + significance + FDR framework. None produced a
reliable positive effect on `vol_regime_h7`. This wasn't a dead end
treated as one, though: the pattern across every attempt (level values,
however filtered or regularized, don't help) pointed toward a different
hypothesis -- **what if the object of interest isn't a single day's
topological snapshot, but the *trajectory* of topology through time?**
That reframing is where the next notebook picks up, and it's where this
project's confirmed findings actually come from.
